In [1]:
import os
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_squared_error
from xgboost import XGBRegressor
from scipy.stats import randint, loguniform


In [2]:
DATA_PATH = "../data/processed/original/original_full.csv"
EMB_PATH = "../outputs/embeddings/original/embeddings_original_full.npy"

In [3]:
df = pd.read_csv(DATA_PATH)
df_emb = np.load(EMB_PATH)

print("df shape:", df.shape)
print("embeddings shape:", df_emb.shape)

assert len(df) == df_emb.shape[0], "df and df_emb must have same #rows"


df shape: (20759, 362)
embeddings shape: (20759, 768)


In [4]:
df = pd.get_dummies(df ,columns=['race_ethnicity', 'gender',
       'grade_level', 'economically_disadvantaged'], drop_first=False)

In [5]:
TARGET_COL = "holistic_essay_score"   # change if needed

# 1) Target
y = df[TARGET_COL]

# 2) Tabular numeric features (drop target & obviously non-features)
drop_cols = [TARGET_COL, 'text', 'holistic_essay_score', 'prompt_name']


x = df.drop(columns=drop_cols, errors="ignore")

X = np.hstack([x, df_emb])

print("X shape:", X.shape)


X shape: (20759, 1139)


In [6]:
# 5-fold CV on ALL essays (no SES split here)
inner_kf = KFold(n_splits=3, shuffle=True, random_state=2025)

# RMSE scorer (negative because sklearn assumes "higher is better")
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

# Hyperparameter search space (baseline)
param_dist = {
    "n_estimators": randint(300, 1001),   # trees 
    "max_depth": randint(3, 9), # depth
    "reg_lambda": loguniform(1e-2, 10.0), # lasso
    "reg_alpha": loguniform(1e-3, 1.0), # ridge
}
param_dist


{'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x171a0d790>,
 'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x172a75be0>,
 'reg_lambda': <scipy.stats._distn_infrastructure.rv_continuous_frozen at 0x172a75bb0>,
 'reg_alpha': <scipy.stats._distn_infrastructure.rv_continuous_frozen at 0x172a75f10>}

In [7]:
base_model = XGBRegressor(
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=12,                  # increase if you want more thorough search
    scoring=rmse_scorer,
    cv=inner_kf,
    n_jobs=-1,
    verbose=2,
    random_state=123,
)

search.fit(X, y)

print("Best params:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)


Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END max_depth=4, n_estimators=396, reg_alpha=0.054838162109857695, reg_lambda=0.026293735380546718; total time=  41.8s
[CV] END max_depth=4, n_estimators=396, reg_alpha=0.054838162109857695, reg_lambda=0.026293735380546718; total time=  42.1s
[CV] END max_depth=3, n_estimators=524, reg_alpha=0.15386828816779854, reg_lambda=0.20687915518060288; total time=  38.0s
[CV] END max_depth=4, n_estimators=396, reg_alpha=0.054838162109857695, reg_lambda=0.026293735380546718; total time=  42.9s
[CV] END max_depth=3, n_estimators=524, reg_alpha=0.15386828816779854, reg_lambda=0.20687915518060288; total time=  38.3s
[CV] END max_depth=4, n_estimators=895, reg_alpha=0.02974108445898576, reg_lambda=2.188181216685915; total time= 1.5min
[CV] END max_depth=4, n_estimators=895, reg_alpha=0.02974108445898576, reg_lambda=2.188181216685915; total time= 1.5min
[CV] END max_depth=4, n_estimators=895, reg_alpha=0.02974108445898576, reg_lambda=2

In [8]:
OUT_DIR = "../outputs/model/parameters/"
os.makedirs(OUT_DIR, exist_ok=True)

best_params = search.best_params_

with open(os.path.join(OUT_DIR, "baseline_best_params.json"), "w") as f:
    json.dump(best_params, f, indent=2)

best_params

{'max_depth': 5,
 'n_estimators': 876,
 'reg_alpha': np.float64(0.0012132376107922202),
 'reg_lambda': np.float64(0.033244480342260616)}